In [1]:
!pip install langchain_huggingface

In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task = "text-generation"
)

model = ChatHuggingFace(llm=llm)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
from langchain_core.prompts import PromptTemplate

In [4]:
# 1st prompt -> detailed report
template1 = PromptTemplate(
    template="Write a detailed report on {topic}",
    input_variables=['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template= "Write a 5 line summary on the following text. /n {text}",
    input_variables=['text']
)


In [5]:
prompt1 = template1.invoke({'topic':'black hole'})

result = model.invoke(prompt1)

In [6]:
prompt2 = template2.invoke({"text":result.content})
result1 = model.invoke(prompt2)

In [7]:
print(result1.content)

<|user|>
Write a 5 line summary on the following text. /n <|user|>
Write a detailed report on black hole</s>
<|assistant|>
Introduction

Black holes are celestial bodies that are so massive that they have a gravitational pull so strong that nothing, not even light, can escape from their influence. These celestial bodies are known to be present in the universe, and they have fascinated scientists and researchers for many years. Black holes are classified based on their mass, which determines their gravitational pull. There are three types of black holes: mini-black holes, intermediate-mass black holes, and supermassive black holes. In this report, we will delve deep into the phenomenon of black holes and the properties that make them unique.

Properties of Black Holes

1. Mass: Black holes are not made of matter but rather matter that is compressed to a point. The mass of a black hole is directly proportional to its size and is given by the relation, M = GM. Here, M is the mass of the b

### StrOutputParser

In [8]:
from langchain_core.output_parsers import StrOutputParser
# 1st prompt -> detailed report
template1 = PromptTemplate(
    template="Write a detailed report on {topic}",
    input_variables=['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template= "Write a 5 line summary on the following text. \n {text}",
    input_variables=['text']
)

In [9]:
parser = StrOutputParser()

In [10]:
chain = template1 | model | parser| template2 | model | parser

In [11]:
result = chain.invoke({'topic':'black hole'})
print(result)

<|user|>
Write a 5 line summary on the following text. 
 <|user|>
Write a detailed report on black hole</s>
<|assistant|>
Introduction:

Black holes are one of the most mysterious objects in the universe. They are regions in space where gravity is so strong that nothing, not even light, can escape. But what exactly is a black hole, and how do they form? In this report, we will explore the origins and properties of black holes, as well as the latest research and developments in understanding them.

Origins and Properties of Black Holes:

Black holes are created when massive stars or other celestial bodies collapse to a point of no return. The collapse releases immense amounts of energy, which creates a singularity, or the point of no return, at the center of the collapsing object. As the energy is released, it creates a region of intense gravity that rapidly contracts until it becomes a singularity. This singularity is the point of no return, and is the point at which space and time can

### JSONOutputParser

In [12]:
from langchain_core.output_parsers import JsonOutputParser
parser = JsonOutputParser()

template = PromptTemplate(
    template = "Give me the name, age and city of a fictional person \n {format_instruction}",
    input_variables = [],
    partial_variables= {'format_instruction': parser.get_format_instructions()}
)

prompt = template.format()

print(prompt)

Give me the name, age and city of a fictional person 
 Return a JSON object.


In [13]:
result = model.invoke(prompt)
result

AIMessage(content='<|user|>\nGive me the name, age and city of a fictional person \n Return a JSON object.</s>\n<|assistant|>\nHere\'s a fictional person named John who is 35 years old and lives in New York City:\n\n```python\nfrom json import loads\n\ndef get_person():\n    person = {\n        "name": "John",\n        "age": 35,\n        "city": "New York City"\n    }\n    return loads(json.dumps(person))\n\nprint(get_person())\n```\n\nOutput:\n\n```json\n{\n    "name": "John",\n    "age": 35,\n    "city": "New York City"\n}\n```', additional_kwargs={}, response_metadata={}, id='lc_run--019d6e22-b658-7f41-bf23-ff6c369232d0-0', tool_calls=[], invalid_tool_calls=[])

In [14]:
final_result = parser.parse(result.content)
final_result

OutputParserException: Invalid json output: <|user|>
Give me the name, age and city of a fictional person 
 Return a JSON object.</s>
<|assistant|>
Here's a fictional person named John who is 35 years old and lives in New York City:

```python
from json import loads

def get_person():
    person = {
        "name": "John",
        "age": 35,
        "city": "New York City"
    }
    return loads(json.dumps(person))

print(get_person())
```

Output:

```json
{
    "name": "John",
    "age": 35,
    "city": "New York City"
}
```
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [15]:
# using chain
chain = template | model | parser

result = chain.invoke({})

result

OutputParserException: Invalid json output: <|user|>
Give me the name, age and city of a fictional person 
 Return a JSON object.</s>
<|assistant|>
Sure, here's an example of a PHP script that retrieves a fictional person's name, age, and city:

```php
<?php

function getFictionalPerson() {
    $url = "https://jsonplaceholder.typicode.com/users";
    $ch = curl_init($url);
    curl_setopt($ch, CURLOPT_RETURNTRANSFER, true);
    $result = curl_exec($ch);
    curl_close($ch);

    $data = json_decode($result, true);

    return [
        'name' => $data[0]['name'],
        'age' => $data[0]['age'],
        'city' => $data[0]['city']
    ];
}

$person = getFictionalPerson();

echo "Name: ". $person['name']. "\n";
echo "Age: ". $person['age']. "\n";
echo "City: ". $person['city']. "\n";

?>
```

In this script, we first retrieve the JSON data from the `https://jsonplaceholder.typicode.com/users` URL. We then parse the JSON data into an array using the `json_decode()` function. We then extr

In [16]:
!pip install langchain

In [18]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [27]:
class Person(BaseModel):

  name : str = Field(description="Name of the person")
  age : int = Field(gt=18, description="Age of the person")
  city : str = Field(description="city where the person belongs to")

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template = "Generate the name, age and city of a fictional {place} person \n {format_instruction}",
    input_variables=["place"],
    partial_variables= {'format_instruction': parser.get_format_instructions()}

)

prompt = template.invoke({'place':"indian"})

print(prompt)
result = model.invoke(prompt)

final_result = parser.parse(result.content)

final_result

text='Generate the name, age and city of a fictional indian person \n The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "Name of the person", "title": "Name", "type": "string"}, "age": {"description": "Age of the person", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "city": {"description": "city where the person belongs to", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```'


OutputParserException: Failed to parse Person from completion {"properties": {"name": {"description": "Name of the person", "title": "Name", "type": "string"}, "age": {"description": "Age of the person", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "city": {"description": "city where the person belongs to", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}. Got: 3 validation errors for Person
name
  Field required [type=missing, input_value={'properties': {'name': {...['name', 'age', 'city']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
age
  Field required [type=missing, input_value={'properties': {'name': {...['name', 'age', 'city']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
city
  Field required [type=missing, input_value={'properties': {'name': {...['name', 'age', 'city']}, input_type=dict]
    For further information visit https://errors.pyd